In [6]:
# Instalação do Java 8 (necessário para Spark)
!apt-get update > /dev/null
!apt-get install openjdk-8-jdk-headless -qq > /dev/null

# Instalação do PySpark
!pip install pyspark --quiet

# Verificação da versão do Java
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
!java -version

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
openjdk version "1.8.0_462"
OpenJDK Runtime Environment (build 1.8.0_462-8u462-ga~us1-0ubuntu2~22.04.2-b08)
OpenJDK 64-Bit Server VM (build 25.462-b08, mixed mode)


In [9]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession

spark = SparkSession.builder.master("local[*]").getOrCreate()

conf = SparkConf().set('spark.ui.port', '4050').setAppName("twitter").setMaster("local[*]")
sc = SparkContext.getOrCreate(conf=conf)

In [10]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [11]:
url = "/content/drive/My Drive/Colab Notebooks/tweets.csv"
df = spark.read.option("inferrSchema", True).option("header", True).csv(url)
df.printSchema()

root
 |-- ItemID: string (nullable = true)
 |-- Sentiment: string (nullable = true)
 |-- SentimentSource: string (nullable = true)
 |-- SentimentText: string (nullable = true)



In [14]:
df.write.saveAsTable("tweets")

In [16]:
spark.sql("SELECT Sentiment, COUNT(*) FROM tweets GROUP BY Sentiment").show()

+---------+--------+
|Sentiment|count(1)|
+---------+--------+
|      pos|   26921|
|        0|       2|
|      neg|   23079|
+---------+--------+



In [17]:
from pyspark.sql import HiveContext
HiveContext = HiveContext(sc)
hiveQuery = "select SentimentText from tweets where Sentiment = 'pos';"
dfPos = df = HiveContext.sql(hiveQuery)
rddPos = dfPos.rdd

/usr/local/lib/python3.12/dist-packages/pyspark/sql/context.py:733: FutureWarning: HiveContext is deprecated in Spark 2.0.0. Please use SparkSession.builder.enableHiveSupport().getOrCreate() instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pyspark/sql/context.py:113: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


In [18]:
stopWords = ['i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you','your', 'yours', 'yourself', 'yourselves', 'he', 'him', 'his', 'himself', 'she', 'her',
             'hers', 'herself', 'it', 'its', 'itself', 'they', 'them', 'their', 'theirs', 'themselves', 'what', 'which', 'who', 'whom', 'this', 'that', 'these', 'those',
             'am', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had', 'having', 'do','does', 'did', 'doing', 'a', 'an', 'the', 'and', 'but', 'if',
             'or', 'because', 'as','until', 'while', 'of', 'at', 'by', 'for', 'with', 'about', 'against', 'between', 'into', 'through', 'during', 'before', 'after', 'above',
             'below', 'to', 'from', 'up', 'down', 'in', 'out', 'on', 'off', 'over', 'under', 'again', 'further', 'then', 'once','here', 'there', 'when', 'where', 'why', 'how',
             'all', 'any', 'both', 'each', 'few','more', 'most', 'other', 'some', 'such', 'no', 'nor', 'not', 'only', 'own', 'same', 'so', 'than', 'too', 'very', 'can', 'will',
             'just', 'don', 't', 's', 'should', 'now']

In [19]:
contadorPalavras = rddPos.map(lambda x:x.SentimentText.replace(',','').replace('.', ' ').replace('-','').lower()) \
 .flatMap(lambda x: x.split()) \
 .filter(lambda x: x not in stopWords) \
 .map(lambda x: (x,1)) \
 .reduceByKey(lambda x,y:x+y) \
 .map(lambda x:(x[1],x[0])) \
 .sortByKey(False)

In [20]:
contadorPalavras.take(10)

[(1839, 'good'),
 (1827, "i'm"),
 (1650, 'love'),
 (1419, 'like'),
 (1250, 'lol'),
 (1231, 'thanks'),
 (1195, 'get'),
 (1186, 'u'),
 (1113, "it's"),
 (1037, 'know')]